In [ ]:
import os, sys, shutil, subprocess

# ---- EDIT THIS if your Drive folder differs ----
DRIVE_ROOT = '/content/drive/MyDrive/thesis_7gcn'
ZIP_NAME   = 'HAABSA7GCN_7gcn.zip'
# -------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(f'{DRIVE_ROOT}/{ZIP_NAME}'), \
    f"zip not found at {DRIVE_ROOT}/{ZIP_NAME} — upload it to Drive first"

# unzip into ephemeral /content (fast local disk for the run)
WORK_BASE = '/content/work'
shutil.rmtree(WORK_BASE, ignore_errors=True)
os.makedirs(WORK_BASE, exist_ok=True)
subprocess.run(['unzip', '-q', f'{DRIVE_ROOT}/{ZIP_NAME}', '-d', WORK_BASE], check=True)

INPUT_BASE = f'{WORK_BASE}/HAABSA7GCN'
assert os.path.isdir(f'{INPUT_BASE}/src'), f"src/ not found under {INPUT_BASE}"
assert os.path.isdir(f'{INPUT_BASE}/data'), f"data/ not found under {INPUT_BASE}"

# output goes to DRIVE (survives disconnects)
TPE_OUT = f'{DRIVE_ROOT}/tpe_output'
os.makedirs(TPE_OUT, exist_ok=True)

print('src/:', sorted(os.listdir(f'{INPUT_BASE}/src'))[:8], '...')
print('INPUT_BASE:', INPUT_BASE)
print('TPE_OUT (Drive):', TPE_OUT)

In [ ]:
import shutil, os, sys

WORK = '/content/run'
os.makedirs(f'{WORK}/data', exist_ok=True)
for f in os.listdir(f'{INPUT_BASE}/data'):
    shutil.copy(f'{INPUT_BASE}/data/{f}', f'{WORK}/data/{f}')
shutil.copy(f'{INPUT_BASE}/cooc_matrix_final2.csv', f'{WORK}/cooc_matrix_final2.csv')
shutil.copy(f'{INPUT_BASE}/src/test_ontology_keys.csv', f'{WORK}/test_ontology_keys.csv')
for m in ['cross_example.py','fusion.py']:
    shutil.copy(f'{INPUT_BASE}/src/{m}', f'{WORK}/{m}')
sys.path.insert(0, WORK)
os.chdir(WORK)
print('cwd:', os.getcwd())
for y in ['2015','2016']:
    assert os.path.exists(f'data/cross_features_{y}.pt')
assert os.path.exists('cross_example.py') and os.path.exists('fusion.py')
print('data + cross_features + modules OK')

In [ ]:
!pip install pytorch_transformers==1.2.0 hyperopt --quiet

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('capability:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'NONE')
import pytorch_transformers; print('pytorch_transformers:', pytorch_transformers.__version__)

In [ ]:
import nbformat
nb = nbformat.read(f'{INPUT_BASE}/src/7GCN.ipynb', as_version=4)
DEFINITION_CELLS = list(range(0, 25))
for i, cell in enumerate(nb.cells):
    if i not in DEFINITION_CELLS or cell.cell_type != 'code':
        continue
    src = ''.join(cell.source)
    if not src.strip():
        continue
    try:
        exec(src, globals())
        print(f"OK cell {i}")
    except Exception as e:
        print(f"FAILED cell {i}: {type(e).__name__}: {e}")
        raise

In [ ]:
import pandas as pd
cooc = pd.read_csv('cooc_matrix_final2.csv', index_col=0)
onto_words_df = pd.read_csv('test_ontology_keys.csv', sep=';')
onto_words = [x for sub in onto_words_df.values.tolist() for x in sub]
onto_words = list(dict.fromkeys(onto_words))
print('cooc:', cooc.shape, '| onto_words:', len(onto_words))

In [ ]:
import os, json, time, pickle, math, shutil, torch, gc
from hyperopt import fmin, tpe, hp, STATUS_OK
from hyperopt.fmin import generate_trials_to_calculate

# ---- search config ----
SEED          = 7
YEAR          = '2015'         
SEARCH_EPOCHS = 8
MAX_TRIALS    = 14
XSIM_TOP_K_FIXED = 20          
PKL  = f'{TPE_OUT}/tpe_7gcn_4dim_{YEAR}_seed{SEED}.pkl'
BEST = f'{TPE_OUT}/tpe_7gcn_4dim_best_so_far.json'
LOCAL_PKL = '/content/tpe_local.pkl'   
WALL_GUARD_SEC = 11.5 * 3600

space = [
    hp.loguniform('lr', math.log(8e-6), math.log(3e-5)),
    hp.quniform('dropout', 0.15, 0.35, 0.05),
    hp.loguniform('l2reg', math.log(5e-4), math.log(2e-2)),
    hp.quniform('fusion_dropout', 0.0, 0.3, 0.1),
]
CONCAT_DROPOUT_FIXED = 0.2285714285714286

# seed first trial with the 4GCN-tuned config + fusion_dropout default
seed_trial = [{'lr': 8.836537235645673e-06, 'dropout': 0.25,
               'l2reg': 0.0005059218096160434, 'fusion_dropout': 0.0}]

t_start = time.time()

def objective(params):
    global opt
    lr, drop, l2, fdrop = params
    if time.time() - t_start > WALL_GUARD_SEC:
        return {'status': STATUS_OK, 'loss': 1.0, 'val_acc': 0.0, 'space': params, 'skipped': True}
    print(f"\n[trial] lr={lr:.3e} drop={drop} l2={l2:.3e} fdrop={fdrop} topk={XSIM_TOP_K_FIXED}(fixed)")
    opt = get_args(
        model_type='tri_gcn',
        tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
        constgcn=True, xcatgcn=True, xsimgcn=True,
        fusion_type='gated', fusion_dropout=float(fdrop), xsim_top_k=XSIM_TOP_K_FIXED,
        year=YEAR, seed=SEED,
        learning_rate=float(lr), dropout=float(drop),
        concat_dropout=CONCAT_DROPOUT_FIXED, l2reg=float(l2),
        num_epoch=SEARCH_EPOCHS, batch_size=4,
        save_models='none', use_ensemble=False,
        cooc=cooc, onto_words=onto_words, do_train=True, do_eval=True,
        path='/content/run/tmp_trial',
    )
    opt.device = torch.device('cuda')
    t0 = time.time()
    max_val_acc, test_acc, test_f1 = main(opt)
    dt = (time.time() - t0) / 60
    print(f"[trial] -> val={max_val_acc:.4f} test={test_acc:.4f} ({dt:.1f} min)")
    gc.collect(); torch.cuda.empty_cache()
    return {'status': STATUS_OK, 'loss': -float(max_val_acc),
            'val_acc': float(max_val_acc), 'test_acc': float(test_acc),
            'test_f1': float(test_f1), 'space': params}

# resume if a pickle already exists (prefer Drive, fall back to local), else fresh start
if os.path.exists(PKL):
    trials = pickle.load(open(PKL, 'rb'))
    done = len([t for t in trials.trials if t['state'] == 2])
    print(f"RESUMING from Drive: {done} trials already done")
elif os.path.exists(LOCAL_PKL):
    trials = pickle.load(open(LOCAL_PKL, 'rb'))
    done = len([t for t in trials.trials if t['state'] == 2])
    print(f"RESUMING from LOCAL: {done} trials already done")
else:
    trials = generate_trials_to_calculate(seed_trial)
    done = 0
    print("FRESH START (seeded with 4GCN-tuned config + fusion_dropout=0)")

# run one trial at a time, HARDENED dual-save (local always + Drive best-effort) after each
for n in range(done, MAX_TRIALS):
    fmin(objective, space, algo=tpe.suggest, max_evals=n + 1,
         trials=trials, show_progressbar=False,
         trials_save_file='')          # '' = hyperopt's "don't save" sentinel (NOT None)

    # --- hardened save: local first (always works), then best-effort Drive copy ---
    pickle.dump(trials, open(LOCAL_PKL, 'wb'))
    try:
        shutil.copy(LOCAL_PKL, PKL)
        assert os.path.exists(PKL)
        print(f"   saved: local + drive ({os.path.getsize(PKL)} bytes)")
    except Exception as e:
        print(f"   LOCAL saved OK, drive copy FAILED (will retry next trial): {e}")

    ok = [t for t in trials.trials if t['state'] == 2 and t['result'].get('val_acc', 0) > 0]
    if ok:
        best = max(ok, key=lambda t: t['result']['val_acc'])
        r = best['result']; lr, drop, l2, fdrop = r['space']
        best_json = {'val_acc': r['val_acc'], 'test_acc': r.get('test_acc'),
                     'lr': float(lr), 'dropout': float(drop), 'l2reg': float(l2),
                     'fusion_dropout': float(fdrop), 'xsim_top_k': XSIM_TOP_K_FIXED,
                     'n_trials_done': len(ok)}
        json.dump(best_json, open('/content/tpe_best_local.json', 'w'), indent=2)   # local mirror
        try:
            json.dump(best_json, open(BEST, 'w'), indent=2)
        except Exception as e:
            print(f"   best.json drive write failed: {e}")
        print(f"== {len(ok)} trials done; best val={r['val_acc']:.4f}")
    if time.time() - t_start > WALL_GUARD_SEC:
        print("WALL GUARD hit — stopping"); break

print("SEARCH LOOP DONE")

In [ ]:
import pickle, os, shutil, time

# 1. LOCAL save first — never depends on Drive
pickle.dump(trials, open('/content/tpe_7gcn_RESCUE.pkl','wb'))
print("local rescue saved:", os.path.getsize('/content/tpe_7gcn_RESCUE.pkl'), "bytes")
print("completed trials:", len([t for t in trials.trials if t['state']==2]))

# 2. print every result to the log (text-safe record, survives anything)
print("\n--- TRIAL RESULTS ---")
for i,t in enumerate(trials.trials):
    if t['state']==2:
        r=t['result']
        print(i, "val=%.4f test=%.4f"%(r.get('val_acc',0), r.get('test_acc',0)), r.get('space'))

# 3. remount Drive cleanly and copy there
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/thesis_7gcn/tpe_output', exist_ok=True)
shutil.copy('/content/tpe_7gcn_RESCUE.pkl', PKL)
time.sleep(5)
print("\non drive now:", os.path.exists(PKL))

# 4. download to laptop — ultimate backup
from google.colab import files
files.download('/content/tpe_7gcn_RESCUE.pkl')

In [ ]:
import pickle
trials = pickle.load(open(PKL, 'rb'))
rows = []
for t in trials.trials:
    if t['state'] != 2: continue
    r = t['result']
    if r.get('val_acc', 0) <= 0: continue
    lr, drop, l2, fdrop = r['space']
    rows.append((r['val_acc'], r.get('test_acc', float('nan')),
                 float(lr), float(drop), float(l2), float(fdrop)))
rows.sort(key=lambda x: -x[0])
print(f"{'val':>7} {'test':>7} {'lr':>10} {'drop':>5} {'l2':>9} {'fdrop':>5}")
for v, te, lr, d, l2, fd in rows:
    print(f"{v:>7.4f} {te:>7.4f} {lr:>10.3e} {d:>5.2f} {l2:>9.3e} {fd:>5.2f}")